# 02 RFM Segmentation, Churn Risk, and Cohort Retention

This notebook builds customer-level analytics tables for segmentation, churn/inactivity risk, and retention analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_PATH = Path("../data/raw/online_retail_II.xlsx")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)

In [ ]:
# Load and clean data. This repeats key steps so the notebook can run independently.
xl = pd.ExcelFile(RAW_PATH)
frames = []
for sheet in xl.sheet_names:
    temp = pd.read_excel(RAW_PATH, sheet_name=sheet)
    temp["source_sheet"] = sheet
    frames.append(temp)
raw = pd.concat(frames, ignore_index=True)

rename_map = {
    "Invoice": "invoice_no", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "invoice_date", "Price": "unit_price",
    "Customer ID": "customer_id", "Country": "country"
}
df = raw.rename(columns=rename_map).copy()
df["invoice_no"] = df["invoice_no"].astype(str)
df["stock_code"] = df["stock_code"].astype(str)
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce")
df["revenue"] = df["quantity"] * df["unit_price"]
df["is_cancelled"] = df["invoice_no"].str.startswith("C") | (df["quantity"] < 0)
df["invoice_month"] = df["invoice_date"].dt.to_period("M").astype(str)

sales_customer = df[(~df["is_cancelled"]) & (df["quantity"] > 0) & (df["unit_price"] > 0) & df["customer_id"].notna()].copy()
sales_customer["customer_id"] = sales_customer["customer_id"].astype(int).astype(str)
print(sales_customer.shape)
sales_customer.head()

In [ ]:
# Build RFM table
snapshot_date = sales_customer["invoice_date"].max() + pd.Timedelta(days=1)

rfm = sales_customer.groupby("customer_id").agg(
    first_purchase_date=("invoice_date", "min"),
    last_purchase_date=("invoice_date", "max"),
    recency_days=("invoice_date", lambda x: (snapshot_date - x.max()).days),
    frequency=("invoice_no", "nunique"),
    monetary_value=("revenue", "sum"),
    unique_products=("stock_code", "nunique"),
    active_months=("invoice_month", "nunique")
).reset_index()

rfm["avg_order_value"] = rfm["monetary_value"] / rfm["frequency"]
rfm["customer_lifetime_days"] = (rfm["last_purchase_date"] - rfm["first_purchase_date"]).dt.days
rfm.head()

In [ ]:
# Add return/cancellation rate by customer
all_customer_lines = df[df["customer_id"].notna()].copy()
all_customer_lines["customer_id"] = all_customer_lines["customer_id"].astype(int).astype(str)

return_rate = all_customer_lines.groupby("customer_id").agg(
    total_lines=("invoice_no", "size"),
    cancelled_lines=("is_cancelled", "sum")
).reset_index()
return_rate["return_rate"] = return_rate["cancelled_lines"] / return_rate["total_lines"]

rfm = rfm.merge(return_rate[["customer_id", "return_rate"]], on="customer_id", how="left")
rfm["return_rate"] = rfm["return_rate"].fillna(0)
rfm.head()

In [ ]:
# RFM scoring. Recency is reversed because lower recency is better.
rfm["r_score"] = pd.qcut(rfm["recency_days"].rank(method="first"), 5, labels=[5,4,3,2,1]).astype(int)
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary_value"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["rfm_score"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)

def assign_segment(row):
    r, f, m = row["r_score"], row["f_score"], row["m_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    if r >= 3 and f >= 4:
        return "Loyal Customers"
    if r >= 4 and f <= 2:
        return "New / Promising"
    if r <= 2 and f >= 4 and m >= 4:
        return "Cannot Lose Them"
    if r <= 2 and (f >= 3 or m >= 3):
        return "At Risk"
    if r == 3 and f >= 3:
        return "Need Attention"
    if r <= 2 and f <= 2:
        return "Hibernating / Lost"
    return "Potential Loyalists"

rfm["customer_segment"] = rfm.apply(assign_segment, axis=1)
rfm["customer_segment"].value_counts()

In [ ]:
# Churn / inactivity labeling
CHURN_THRESHOLD_DAYS = 90
rfm["churn_risk_label"] = (rfm["recency_days"] > CHURN_THRESHOLD_DAYS).astype(int)
rfm["churn_status"] = np.where(rfm["churn_risk_label"] == 1, "Inactive / Churn Risk", "Active")

segment_summary = rfm.groupby("customer_segment").agg(
    customers=("customer_id", "nunique"),
    revenue=("monetary_value", "sum"),
    avg_recency_days=("recency_days", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary_value=("monetary_value", "mean"),
    churn_risk_rate=("churn_risk_label", "mean")
).reset_index().sort_values("revenue", ascending=False)

segment_summary

In [ ]:
# High-value inactive customers for targeted reactivation
high_value_churn_risk = rfm[rfm["churn_risk_label"] == 1].sort_values("monetary_value", ascending=False)
high_value_churn_risk.head(20)

In [ ]:
# Cohort retention matrix
cohort_data = sales_customer.copy()
cohort_data["order_month"] = cohort_data["invoice_date"].dt.to_period("M")
cohort_data["cohort_month"] = cohort_data.groupby("customer_id")["order_month"].transform("min")
cohort_data["cohort_index"] = (
    (cohort_data["order_month"].dt.year - cohort_data["cohort_month"].dt.year) * 12
    + (cohort_data["order_month"].dt.month - cohort_data["cohort_month"].dt.month)
)

cohort_counts = cohort_data.groupby(["cohort_month", "cohort_index"])["customer_id"].nunique().reset_index()
cohort_pivot = cohort_counts.pivot(index="cohort_month", columns="cohort_index", values="customer_id")
cohort_retention = cohort_pivot.divide(cohort_pivot.iloc[:, 0], axis=0).round(4)
cohort_retention.index = cohort_retention.index.astype(str)
cohort_retention.head()

In [ ]:
# Save outputs
rfm.to_csv(PROCESSED_DIR / "customer_rfm_segments.csv", index=False)
rfm[[
    "customer_id", "recency_days", "frequency", "monetary_value", "avg_order_value",
    "return_rate", "customer_segment", "churn_risk_label", "churn_status"
]].to_csv(PROCESSED_DIR / "customer_churn_labels.csv", index=False)
segment_summary.to_csv(PROCESSED_DIR / "segment_summary.csv", index=False)
cohort_retention.to_csv(PROCESSED_DIR / "cohort_retention_matrix.csv")
high_value_churn_risk.head(100).to_csv(PROCESSED_DIR / "high_value_churn_risk_customers.csv", index=False)

print("Saved RFM, churn, and cohort outputs.")

In [ ]:
# Segment revenue chart
segment_plot = segment_summary.sort_values("revenue")
plt.figure(figsize=(10, 5))
plt.barh(segment_plot["customer_segment"], segment_plot["revenue"])
plt.title("Revenue by Customer Segment")
plt.xlabel("Revenue")
plt.tight_layout()
plt.show()